In [1]:
import pandas as pd
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import (
    Augment,
    DummyMatrix,
    Gram,
    HadamardProduct,
    IdentityMatrix,
    KroneckerProduct,
    LowerTriangularMatrix,
    ScalarMatrix,
)

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=3, sci_mode=False)
pd.set_option("display.precision", 3)

generator = torch.Generator().manual_seed(2026)
n_subjects, n_days = 72, 8

subject = torch.arange(n_subjects).repeat_interleave(n_days)
day = torch.arange(n_days).repeat(n_subjects)

z_intercept = torch.randn(n_subjects, generator=generator)
z_slope = torch.randn(n_subjects, generator=generator)
b_intercept = 30.0 * z_intercept
b_slope = 10.0 * (-0.5 * z_intercept + 0.75**0.5 * z_slope)

y = (
    250.0
    + 10.0 * day
    + b_intercept[subject]
    + b_slope[subject] * day
    + 25.0 * torch.randn(len(day), generator=generator)
)

data = pd.DataFrame(
    {
        "subject": [f"subject_{i:02d}" for i in subject.tolist()],
        "day": day.numpy(),
        "y": y.numpy(),
    }
)

y = torch.tensor(data["y"].values)
X = torch.stack(
    [
        torch.ones(len(data)),
        torch.tensor(data["day"].values, dtype=torch.float64),
    ],
    dim=1,
)
data.head()

,subject,day,y
0,subject_00,0,315.261
1,subject_00,1,256.463
2,subject_00,2,299.845
3,subject_00,3,232.774
4,subject_00,4,313.987


In [2]:
data.describe(include="all").T[
    ["count", "unique", "top", "freq", "mean", "std", "min", "max"]
]

,count,unique,top,freq,mean,std,min,max
subject,576,72,subject_00,8,NaN,NaN,NaN,NaN
day,576.0,NaN,NaN,NaN,3.5,2.293,0.0,7.0
y,576.0,NaN,NaN,NaN,291.364,50.733,156.317,462.631


In [3]:
D = DummyMatrix(data["subject"])
day_values = torch.tensor(data["day"].values).to(torch.float64).unsqueeze(-1)
Z_slope = HadamardProduct(group=D, x=day_values)

reml_slope = MixedModelREML(subject=(Z_slope, ScalarMatrix(data["subject"])))
reml_slope.optimize(y, X)

reml_slope.build_params(out_format="dict")

{'G/sigma^2': tensor([68.479]), 'R/sigma^2': tensor([877.482])}

In [4]:
Z_subject = Augment(
    intercept=D,
    slope=HadamardProduct(group=D, x=day_values),
)

G_subject = KroneckerProduct(
    covariance=Gram(
        LowerTriangularMatrix(2, 2),
        gram_type="xxt",
    ),
    subject=IdentityMatrix(n_subjects),
)

reml_subject = MixedModelREML(subject=(Z_subject, G_subject))
reml_subject.optimize(y, X)

reml_subject.build_params(out_format="dict")

{'G/covariance/op_0/L_0_0': tensor([29.508]),
 'G/covariance/op_0/L_1_0': tensor([-3.076]),
 'G/covariance/op_0/L_1_1': tensor([7.912]),
 'R/sigma^2': tensor([582.510])}

In [5]:
reml_subject.tree()["G/covariance"]

tensor([[870.723, -90.771],
        [-90.771,  72.065]])

In [6]:
reml_independent = MixedModelREML(
    subject_intercept=(D, ScalarMatrix(data["subject"])),
    subject_slope=(Z_slope, ScalarMatrix(data["subject"])),
)
reml_independent.optimize(y, X)

reml_independent.build_params(out_format="dict")

{'G/subject_intercept/sigma^2': tensor([754.810]),
 'G/subject_slope/sigma^2': tensor([63.209]),
 'R/sigma^2': tensor([593.350])}

In [7]:
print("converged:", reml_subject.is_converged())
print("loglik:", reml_subject.loglik()[0])
print("fixed effects:", reml_subject.blue())
print("fixed-effect SEs:", torch.sqrt(reml_subject.var_blue()))

blups = reml_subject.blup_per_term()["subject"]
print("first three subject BLUPs (intercept, slope):")
print(blups.reshape(2, n_subjects).T[:3])

converged: True
loglik: tensor(-2275.133)
fixed effects: tensor([256.760,   9.887])
fixed-effect SEs: tensor([3.932, 1.092])
first three subject BLUPs (intercept, slope):
tensor([[ 10.326,   0.267],
        [ 18.582, -15.365],
        [-11.472,  15.957]])
